In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

STORAGE_ACCOUNT = "vjretailflow01"
STORAGE_KEY = dbutils.secrets.get(scope="retailflow", key="adls-key")

spark.conf.set(
    f"fs.azure.account.key.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    STORAGE_KEY
)

BRONZE_PATH = f"abfss://bronze@{STORAGE_ACCOUNT}.dfs.core.windows.net"
SILVER_PATH = f"abfss://silver@{STORAGE_ACCOUNT}.dfs.core.windows.net"

print("Setup complete")

In [0]:
orders_bronze = spark.read.format("delta") \
                    .load(f"{BRONZE_PATH}/orders/")

orders_bronze.printSchema()

In [0]:
def transform_orders(df):
    return df \
        .filter(F.col("order_id").isNotNull()) \
        .filter(F.col("customer_id").isNotNull()) \
        .withColumn("order_id", F.col("order_id").cast(T.StringType())) \
        .withColumn("customer_id", F.col("customer_id").cast(T.StringType())) \
        .withColumn("order_status", F.col("order_status").cast(T.StringType())) \
        .withColumn("order_purchase_timestamp", F.col("order_purchase_timestamp").cast(T.TimestampType())) \
        .withColumn("order_approved_at", F.col("order_approved_at").cast(T.TimestampType())) \
        .withColumn("order_delivered_carrier_date", F.col("order_delivered_carrier_date").cast(T.TimestampType())) \
        .withColumn("order_delivered_customer_date", F.col("order_delivered_customer_date").cast(T.TimestampType())) \
        .withColumn("order_estimated_delivery_date", F.col("order_estimated_delivery_date").cast(T.TimestampType())) \
        .withColumn("order_purchase_date", F.to_date(F.col("order_purchase_timestamp"))) \
        .withColumn("delivery_delay_days", 
            F.datediff(
                F.col("order_delivered_customer_date"),
                F.col("order_estimated_delivery_date")
            )
        ) \
        .withColumn("is_late_delivery",
            F.when(F.col("delivery_delay_days") > 0, True).otherwise(False)
        ) \
        .withColumn("_silver_processed_at", F.current_timestamp()) \
        .drop("_ingested_at", "_source_file", "_ingestion_date")

In [0]:
orders_silver = transform_orders(orders_bronze)

print(f"Total rows: {orders_silver.count():,}")
print(f"Null order ids: {orders_silver.filter(F.col('order_id').isNull()).count()}")
print(f"Late deliveries: {orders_silver.filter(F.col("is_late_delivery")).count()}")

print(f"Order_statuses:")
orders_silver.groupBy("order_status").count().orderBy(F.desc("count")).show()

orders_silver.write.format("delta") \
    .mode("overwrite") \
    .partitionBy("order_purchase_date") \
    .save(f"{SILVER_PATH}/orders/")

print("Silver orders writted successfully!")

In [0]:
dbutils.fs.ls(f"{SILVER_PATH}/orders/")

In [0]:
customers_bronze = spark.read.format("delta") \
                             .load(f"{BRONZE_PATH}/customers/")

customers_bronze.printSchema()

In [0]:
def transform_customer(df):
    return df \
        .filter(F.col("customer_id").isNotNull()) \
        .filter(F.col("customer_unique_id").isNotNull()) \
        .withColumn("customer_id", F.col("customer_id").cast(T.StringType())) \
        .withColumn("customer_unique_id", F.col("customer_unique_id").cast(T.StringType())) \
        .withColumn("customer_zip_code_prefix", F.col("customer_zip_code_prefix").cast(T.StringType())) \
        .withColumn("customer_city", F.initcap(F.col("customer_city"))) \
        .withColumn("customer_state", F.upper(F.col("customer_state"))) \
        .withColumn("_customer_silver_processed_at", F.current_timestamp()) \
        .drop("_ingested_at", "_source_file", "_ingestion_date")

customers_silver = transform_customer(customers_bronze)

print(f"Total customers: {customers_silver.count()}")
print(f"Unique customers: {customers_silver.select("customer_unique_id").distinct().count()}")

print("\nTop 5 states by orders")
customers_silver.groupBy("customer_state") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(5)

In [0]:
orders_enriched = orders_silver.join(
    customers_silver,
    on="customer_id",
    how="left"
)

print(f"Total rows: {orders_silver.count():,}")
print(f"Orders enriched count: {orders_enriched.count():,}")
print(f"Nulls in customer state: {orders_enriched.filter(F.col('customer_state').isNull()).count()}")

orders_enriched.write.format("delta") \
    .mode("overwrite") \
    .partitionBy("order_purchase_date") \
    .save(f"{SILVER_PATH}/orders_enriched/")

print("Silver orders enriched writted successfully!")
dbutils.fs.ls(f"{SILVER_PATH}/orders_enriched/")